# AI-Based Constrained Payment Routing Optimization

### FlexFactor — final technical showcase

The project is presented as a sequence of engineering decisions:

> **Start with the simplest formulation. Add complexity only when the simpler formulation fails.**

\[
\text{Prediction}
\rightarrow
\text{Dynamic State}
\rightarrow
\text{Constrained Optimization}
\rightarrow
\text{Online Decisioning}
\rightarrow
\text{Validation}
\]

The goal is not merely to predict transaction success. The goal is to make a **capacity-aware financial routing decision in a non-stationary environment**.

## 0. Setup — public GitHub repo + Google Drive

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, importlib

REPO_URL = "https://github.com/orankedem/flexfactor-routing-final-project.git"
LOCAL_REPO = Path("/content/flexfactor-routing-final-project")

# Colab is disposable; GitHub is the code source of truth.
# This cell is intentionally idempotent: it can be rerun safely.
os.chdir("/content")

# Remove cached imports from a previous clone before replacing its files.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

if LOCAL_REPO.exists():
    print("Removing previous Colab repository copy...")
    shutil.rmtree(LOCAL_REPO)

print("Cloning latest GitHub version...")
result = subprocess.run(
    ["git", "clone", REPO_URL, str(LOCAL_REPO)],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    raise RuntimeError("Git clone failed:\n\n" + result.stderr)

os.chdir(LOCAL_REPO)

if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))

PROJECT_ROOT = LOCAL_REPO

print("✓ Repository ready")
print("Working directory:", os.getcwd())
print("src exists:", (PROJECT_ROOT / "src").exists())
print("artifacts exists:", (PROJECT_ROOT / "artifacts").exists())


In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

DRIVE_ROOT = Path("/content/gdrive/MyDrive")
print("Drive mounted:", DRIVE_ROOT.exists())

## 1. Locate the real project artifacts

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.showcase import (
    discover_artifacts,
    artifact_audit,
    export_exact_configs,
    reliability_table,
    load_candidate_universe,
    detailed_pressure_trace,
    recompute_a1_candidate_probabilities,
)

ART = discover_artifacts(DRIVE_ROOT)
audit = artifact_audit(ART)
display(audit)

assert ART["data"] is not None, "Standardized data not found in Drive"
print("✓ Artifact discovery complete")

# Part I — The financial decision problem

FlexFactor attempts to recover declined card transactions through alternative payment routes. A route is a processor/provider/sponsor-bank combination.

For transaction $i$, candidate route $r$, and time $t$:

$$
\hat p_{ir,t}=P(\text{success}\mid X_i,r,H_t,\text{prior attempt state}).
$$

The goal is not merely accurate classification. The system must **choose a route online** while respecting finite route capacity.

## 2. Load the actual standardized attempt-level data

In [ ]:
attempts = pd.read_parquet(ART["data"])

# Create canonical showcase aliases WITHOUT renaming/removing original columns.
# The saved checkpoint legitimately contains several route representations.
attempts["timestamp"] = pd.to_datetime(attempts["auth_timestamp"], utc=True)
attempts["transaction_id"] = attempts["logical_transaction_id"]
attempts["attempt"] = attempts["attempt_no"]
attempts["amount"] = attempts["amount_numeric"]
attempts["showcase_route"] = attempts["route_clean"]
attempts["merchant_id"] = attempts["merchant"]
attempts["issuer_name"] = attempts["BinCheck_IssuerName"]
attempts["card_network"] = attempts["BinCheck_CardNetwork"]
attempts["card_type"] = attempts["BinCheck_CardType"]
attempts["card_level"] = attempts["BinCheck_CardLevel"]
attempts["mcc"] = attempts["PaymentProvider_MCC"]
attempts["processor"] = attempts["PaymentProvider_Processor"]
attempts["provider"] = attempts["PaymentProvider_Provider"]
attempts["sponsor_bank"] = attempts["PaymentProvider_SponsorBank"]
attempts["response_code"] = attempts["provider_response_code_str"]
attempts["response_description"] = attempts["provider_response_desc_clean"]

stage_summary = (
    attempts.groupby("attempt")
    .agg(
        rows=("success", "size"),
        success_rate=("success", "mean"),
        routes=("showcase_route", "nunique"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
    )
    .reset_index()
)

print(f"Attempt rows: {len(attempts):,}")
print(f"Logical transactions: {attempts['transaction_id'].nunique():,}")
display(stage_summary)

# Part II — Why routing is non-static

Two questions must be answered before building the optimizer:

1. Does route choice contain useful conditional signal?
2. Does the environment drift over time?

The second question changes the architecture fundamentally: if success behavior changes, a static merchant/issuer/route representation is insufficient.

## 3. Concept drift — observed approval rate through time

In [ ]:
a1=attempts[attempts['attempt'].eq(1)].copy()
a1['month']=a1['timestamp'].dt.strftime('%Y-%m')
monthly=a1.groupby('month').agg(rows=('success','size'),success_rate=('success','mean')).reset_index()
display(monthly)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(monthly['month'],monthly['success_rate'],marker='o')
ax.set(title='A1 approval rate changes over time',xlabel='Month',ylabel='Observed success rate')
ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

## 4. Route signal after basic traffic-mix adjustment

In [ ]:
# Descriptive/predictive diagnostic — NOT a causal estimate.
tmp = a1[["merchant_id", "showcase_route", "month", "success", "amount", "amount_decile"]].copy()
keys = ["merchant_id", "month", "amount_decile"]
tmp["group_baseline"] = tmp.groupby(keys, observed=True)["success"].transform("mean")
tmp["residual"] = tmp["success"] - tmp["group_baseline"]

route_res = (
    tmp.groupby("showcase_route")
    .agg(
        rows=("success", "size"),
        raw_success=("success", "mean"),
        residual=("residual", "mean"),
    )
    .query("rows >= 1000")
    .sort_values("residual", ascending=False)
)
display(route_res.head(12))

# Part III — Model selection: why XGBoost?

The project compared model families on the **same frozen A1 architecture and chronological development folds** before tuning the winner.

- CatBoost: native categorical handling
- LightGBM: efficient gradient boosting baseline
- XGBoost: final selected family
- linear SGD/logistic model: sanity baseline

The notebook loads the original comparison table from Drive when available, rather than inventing a comparison after the fact.

## 5. Original model-family comparison

In [ ]:
model_sel = ART["a1_model_selection"]
arch_path = (model_sel / "summary/architecture_aggregate.csv") if model_sel else None
walk_path = (model_sel / "summary/architecture_walk_forward.csv") if model_sel else None

if arch_path and arch_path.exists():
    arch = pd.read_csv(arch_path)
    print("Same-feature, same chronological folds, default-configuration family comparison:")
    display(arch)
    if walk_path and walk_path.exists():
        walk = pd.read_csv(walk_path)
        print("Per-fold results:")
        display(walk)
else:
    print("Architecture comparison checkpoint not found.")
    print("Historical development path: LightGBM feasibility → CatBoost/history experiments → final tuned XGBoost.")

# Part IV — Giving a tabular model memory

A static model sees $X_t$ but not what has been happening recently. The first solution was fixed rolling windows. Their weakness is an arbitrary cutoff: an observation just inside the window receives full weight, while one just outside disappears.

Instead, historical relevance fades continuously:

$$w(\Delta t)=0.5^{\Delta t/h}.$$

Multiple half-lives $h\in\{3,14,60,180\}$ expose several speeds of change to XGBoost.

## 6. Continuous-memory intuition

In [ ]:
ages=np.arange(0,181)
fig,ax=plt.subplots(figsize=(8,4))
for h in [3,14,60,180]:
    ax.plot(ages,0.5**(ages/h),label=f'{h}d half-life')
ax.set(title='Continuous temporal memory',xlabel='Age of historical observation (days)',ylabel='Weight')
ax.legend(); plt.tight_layout(); plt.show()

### Leakage rule

For a transaction at time $t$, historical state is emitted **before** adding outcomes at $t$. Thus all temporal features obey the information set available at decision time. A2 may additionally use the observed A1 response; A3 may use A1 and A2 state.

# Part V — Final A1/A2/A3 predictive models

## 7. Factual chronological predictive performance

In [ ]:
predictive_path = PROJECT_ROOT / "artifacts" / "reference_predictive_metrics.csv"

if predictive_path.exists():
    predictive = pd.read_csv(predictive_path)
else:
    # Frozen final June metrics fallback.
    predictive = pd.DataFrame([
        ["A1", 40545, 0.100555, 0.796946, 0.323912, 0.170765, 0.078848, 0.005462],
        ["A2", 24993, 0.050774, 0.794096, 0.211955, 0.151439, 0.044039, 0.001914],
        ["A3", 8988, 0.019582, 0.870877, 0.137020, 0.206922, 0.018028, 0.002640],
    ], columns=[
        "stage","rows","success_rate","roc_auc","average_precision",
        "logloss_skill","brier","ece"
    ])
display(predictive.style.format({
    'success_rate':'{:.2%}','roc_auc':'{:.3f}','average_precision':'{:.3f}',
    'logloss_skill':'{:.1%}','brier':'{:.4f}','ece':'{:.4f}'
}))

These are **factual observed-route metrics**: outcomes actually occurred. They validate ranking and probability quality separately from any counterfactual routing claim.

## 7.1 Calibration / reliability on untouched June predictions

Because the optimizer uses the **magnitude** of \(\hat p\), not only rank, probability calibration matters directly to the financial decision layer.

A reliability curve compares:

\[
\text{mean predicted probability}
\quad\text{vs}\quad
\text{observed success rate}
\]

on the route that was actually attempted.

In [ ]:
stage_prediction_paths = {
    "A1": ART.get("a1_predictions"),
    "A2": ART.get("a2_predictions"),
    "A3": ART.get("a3_predictions"),
}

calibration_tables = {}
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Perfect calibration")

for stage, path in stage_prediction_paths.items():
    if path is None or not Path(path).exists():
        print("Missing prediction checkpoint:", stage, path)
        continue
    pred = pd.read_parquet(path)
    y_col = "y" if "y" in pred.columns else "success"
    p_col = "p" if "p" in pred.columns else "model_score"
    rel = reliability_table(pred[y_col], pred[p_col], n_bins=10)
    calibration_tables[stage] = rel
    ax.plot(rel["predicted_mean"], rel["observed_rate"], marker="o", label=stage)

ax.set_title("Reliability diagram — frozen June predictions")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed success rate")
ax.legend()
plt.tight_layout()
plt.show()

for stage, rel in calibration_tables.items():
    print(stage)
    display(rel)

# Part VI — What did the models actually use?

## 8. Feature-family importance across A1/A2/A3

In [ ]:
# Prefer the exact saved feature-importance artifact. Fall back to the frozen
# headline table copied from the completed experiment.
fi_roots=[p for p in [ART['feature_policy'], DRIVE_ROOT/'flexfactor_policy_v2/feature_importance_lp_online_v1'] if p]
fi_path=None
for r in fi_roots:
    for rel in ['feature_importance/feature_family_importance.csv','feature_importance/feature_family_importance.csv']:
        p=Path(r)/rel
        if p.exists(): fi_path=p; break
    if fi_path: break

if fi_path:
    fi=pd.read_csv(fi_path)
    display(fi)
else:
    fi=pd.read_csv(PROJECT_ROOT / "artifacts" / "reference_feature_family_importance.csv")
    display(fi)

# Clean visual using the frozen family summary.
ref_fi=pd.read_csv(PROJECT_ROOT / "artifacts" / "reference_feature_family_importance.csv")
for stage in ['A1','A2','A3']:
    g=ref_fi[ref_fi.stage.eq(stage)].sort_values('normalized_gain_pct')
    fig,ax=plt.subplots(figsize=(7,3.6))
    ax.barh(g['family'],g['normalized_gain_pct'])
    ax.set(title=f'{stage} — XGBoost importance by feature family',xlabel='Normalized total gain (%)')
    plt.tight_layout(); plt.show()

## 9. Top individual features from the frozen models

In [ ]:
top_path = None
for r in fi_roots:
    for rel in [
        "feature_importance/specific_feature_importance.csv",
        "feature_importance/feature_importance_all.csv",
    ]:
        p = Path(r) / rel
        if p.exists():
            top_path = p
            break
    if top_path:
        break

specific_importance = pd.DataFrame()
if top_path:
    raw_fi = pd.read_csv(top_path)
    specific_importance = raw_fi.copy()
    for attempt_no in [1, 2, 3]:
        g = raw_fi[raw_fi["attempt"].eq(attempt_no)].copy()
        sort_col = "rank" if "rank" in g.columns else (
            "importance_pct" if "importance_pct" in g.columns else "normalized_total_gain"
        )
        ascending = sort_col == "rank"
        g = g.sort_values(sort_col, ascending=ascending).head(15)
        print(f"A{attempt_no} top individual features")
        cols = [c for c in [
            "rank", "feature", "family", "importance_pct",
            "normalized_total_gain", "total_gain", "average_gain", "split_uses"
        ] if c in g.columns]
        display(g[cols])
else:
    print("Exact individual-feature importance CSV not found; family-level importance remains available.")

# Part VII — Exact frozen configuration audit

The reported results are linked to the **actual saved A1/A2/A3 metadata** rather than approximate hyperparameters typed into the showcase.

The notebook exports compact GitHub-friendly configs containing:

- exact XGBoost parameters;
- exact final tree count;
- exact feature list;
- categorical/numeric schema;
- stage architecture;
- historical half-lives and leakage rules.

Large categorical-level maps remain with the Drive model artifacts.

In [ ]:
exported = export_exact_configs(ART, LOCAL_REPO / "configs")

for stage, path in exported.items():
    print(stage, "->", path)

for stage in ["a1", "a2", "a3"]:
    p = LOCAL_REPO / "configs" / f"{stage}_final.json"
    if not p.exists():
        continue
    cfg = json.load(open(p))
    print("\n", stage.upper())
    display(pd.DataFrame([{
        "family": cfg.get("family"),
        "architecture": cfg.get("architecture"),
        "iterations": cfg.get("iterations"),
        "feature_count": cfg.get("feature_count"),
        "categorical_features": cfg.get("categorical_feature_count"),
        "numeric_features": cfg.get("numeric_feature_count"),
        "final_test_month": cfg.get("final_test_month"),
    }]))
    print("Exact parameters:")
    print(json.dumps(cfg.get("params"), indent=2))

# Part VIII — Final system architecture

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")

def add_box(x, y, w, h, text):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        fill=False, linewidth=1.5,
    )
    ax.add_patch(patch)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=10)
    return (x, y, w, h)

def right_arrow(a, b):
    ax.add_patch(FancyArrowPatch(
        (a[0]+a[2], a[1]+a[3]/2),
        (b[0], b[1]+b[3]/2),
        arrowstyle="->", mutation_scale=14,
    ))

tx = add_box(.3, 4.7, 1.7, 1.1, "Transaction\ncontext")
hist = add_box(.3, 2.3, 1.7, 1.2, "Historical state\n3/14/60/180d")
elig = add_box(2.7, 4.1, 1.8, 1.2, "Eligible\ncandidate routes")
model = add_box(5.2, 4.1, 2.0, 1.2, "A1 / A2 / A3\nXGBoost")
prob = add_box(7.9, 4.1, 1.6, 1.2, "P(success)")
policy_box = add_box(10.2, 4.1, 1.8, 1.2, "Capacity-aware\npressure policy")
choice = add_box(12.5, 4.1, 1.2, 1.2, "Chosen\nroute")
outcome = add_box(10.2, 1.2, 1.8, 1.0, "Observed outcome")
lp = add_box(7.7, 6.3, 2.4, 1.0, "Offline LP oracle\n(full hindsight)")

for a,b in [(tx,elig),(elig,model),(model,prob),(prob,policy_box),(policy_box,choice)]:
    right_arrow(a,b)

ax.add_patch(FancyArrowPatch(
    (hist[0]+hist[2], hist[1]+hist[3]/2),
    (model[0], model[1]+.25), arrowstyle="->", mutation_scale=14
))
ax.add_patch(FancyArrowPatch(
    (prob[0]+prob[2]/2, prob[1]+prob[3]),
    (lp[0]+lp[2]/2, lp[1]), arrowstyle="->", mutation_scale=14
))
ax.add_patch(FancyArrowPatch(
    (choice[0]+choice[2]/2, choice[1]),
    (outcome[0]+outcome[2]/2, outcome[1]+outcome[3]),
    arrowstyle="->", mutation_scale=14
))
ax.add_patch(FancyArrowPatch(
    (outcome[0], outcome[1]+outcome[3]/2),
    (hist[0]+hist[2]/2, hist[1]),
    connectionstyle="arc3,rad=-0.25", arrowstyle="->", mutation_scale=14
))

ax.set_title("FlexFactor final system architecture", fontsize=15)
plt.tight_layout()
plt.show()

# Part IX — Prediction is not the decision

If capacity were unlimited, choose the route with the highest $\hat p$. But route volumes may move only around ±30% from historical volume. This couples decisions across transactions.

The offline LP solves the entire period jointly:

$$\max_x\sum_{i,r}x_{ir}\hat p_{ir}$$

or, for approved transaction value:

$$\max_x\sum_{i,r}x_{ir}Amount_i\hat p_{ir}.$$

## 10. LP oracle — absolute meaning of the 100% benchmark

In [ ]:
policy_path = PROJECT_ROOT / "artifacts" / "reference_policy_results.csv"

if policy_path.exists():
    policy = pd.read_csv(policy_path)
else:
    # Frozen headline policy results fallback.
    policy = pd.DataFrame([
        ["greedy", 0.00, 87.3506, 0.462719, 599132, 0.420790],
        ["success_pressure", 0.10, 162.606, 0.861364, 1067185, 0.749519],
        ["balanced_success_pressure", 0.15, 161.694, 0.856533, 1084469, 0.761658],
        ["value_pressure", 0.20, 140.055, 0.741909, 1212111, 0.851305],
        ["LP_success", None, 188.777, 1.0, 1299129, None],
        ["LP_value", None, 173.116, None, 1423827, 1.0],
    ], columns=[
        "policy","lambda","expected_added_approvals",
        "success_opportunity_capture","expected_added_approved_value",
        "value_opportunity_capture"
    ])

display(policy)
print('100% LP success opportunity = +188.777 model-implied approvals relative to historical routing.')
print('Success-optimal LP also adds ≈1.299M expected approved transaction value.')
print('Value-optimal LP adds ≈1.424M expected approved transaction value.')

**Important:** “100%” means the maximum model-implied benefit found by the full-hindsight LP under the frozen probability model and capacity constraints. It is not 100% transaction success and it is not causal proof.

# Part X — Online routing: greedy vs capacity-aware pressure

Greedy ignores future scarcity. The pressure policy gives an over-used route a dynamic shadow price:

$$Pressure_{r,t}=\frac{A_{r,t}-B_{r,t}}{\max(0.3B_{r,t},1)}$$

$$Score_{ir}=\hat p_{ir}-\lambda Pressure_{r,t}.$$

The adjusted score chooses the route; evaluation still uses the **raw calibrated model probability**.

## 11. Opportunity capture and capacity behavior

In [ ]:
show=policy[policy.policy.isin(['greedy','success_pressure','LP_success'])].copy()
fig,ax=plt.subplots(figsize=(7.5,4))
ax.bar(show['policy'],100*show['success_capture'])
ax.set(title='Share of model-implied LP success opportunity captured',ylabel='Opportunity captured (%)',ylim=(0,105))
plt.tight_layout(); plt.show()

capacity=pd.DataFrame({
    'policy':['Greedy','Success pressure λ=0.10','Balanced pressure λ=0.15','Value pressure λ=0.20'],
    'routes_near_30pct_boundary':[9,1,1,1],
    'mean_abs_route_deviation':[.2636,.1026,.0760,.0753]
})
display(capacity)

# Part XI — Detailed real inference walkthrough

This section uses the **actual saved candidate predictions from the June chronological backtest**. It automatically finds an event where the highest raw-probability route is *not* the pressure-policy choice, then reconstructs the decision from the cumulative route state immediately before that event.

This is the clearest demonstration of the difference between **prediction** and **decision optimization**.

## 12. Load actual candidate scores and explain one decision

In [ ]:
assert ART["backtest"] is not None, "Backtest candidate checkpoints not found in Drive"

# Use A1 so the exact frozen feature matrix/model can also be re-run below.
candidates = load_candidate_universe(
    ART["backtest"],
    month="2026-06",
    attempts=(1,),
)
print(f"A1 candidate rows loaded: {len(candidates):,}")

meta, decision_table = detailed_pressure_trace(
    candidates,
    lambda_=0.10,
    attempt_filter=1,
)

print("Selected real event")
display(pd.DataFrame([meta]))

cols = [c for c in [
    "route", "raw_p_success", "baseline_cumulative",
    "policy_cumulative_before", "pressure", "adjusted_score",
    "feasible", "is_logged_route", "raw_greedy_selected", "selected",
    "_candidate_id",
] if c in decision_table.columns]

display(decision_table[cols].style.format({
    "raw_p_success": "{:.4f}",
    "pressure": "{:+.3f}",
    "adjusted_score": "{:.4f}",
}))

Read the table left to right:

1. **raw_p_success** — what the frozen XGBoost model believes about each candidate route;
2. **baseline / policy cumulative counts** — current route usage state;
3. **pressure** — scarcity/over-use signal;
4. **adjusted_score** — probability minus shadow price;
5. **feasible** — whether the strict development capacity guard allows the choice;
6. **selected** — route chosen online.

The important point is that the model does not directly “choose” the route. It provides calibrated probabilities to a separate constrained decision layer.

## 12.0 Transaction context at decision time

Before looking inside XGBoost, show the underlying transaction that produced this candidate set. This separates **transaction-level context** from the **route-specific state** that changes across candidate rows.

In [ ]:
raw_tx_id = meta.get("TransactionId")

if raw_tx_id is not None:
    tx = attempts[
        attempts["TransactionId"].astype(str).eq(str(raw_tx_id))
        & attempts["attempt"].eq(1)
    ].copy()
else:
    tx = pd.DataFrame()

if tx.empty:
    print("Raw TransactionId was not carried into this candidate checkpoint; event metadata is shown above.")
else:
    context_cols = [c for c in [
        "TransactionId", "timestamp", "amount", "merchant_id", "issuer_name",
        "card_network", "card_type", "card_level", "mcc",
        "processor", "provider", "sponsor_bank", "showcase_route", "success",
    ] if c in tx.columns]
    display(tx[context_cols].head(1))

## 12.1 Re-run the exact frozen XGBoost inference

The chronological backtest stores a compact candidate table, so the
notebook first recovers the original candidate IDs from the event key
and route.

It then loads the exact candidate feature rows and the frozen A1
XGBoost model.

Because the model uses native categorical features, inference must
preserve the **training-time categorical vocabulary**. The replay reads
that vocabulary directly from the serialized XGBoost model. Any raw
category unseen during training is mapped to the model's explicit
`__OTHER__` category when available, otherwise to missing.

The final table compares the saved backtest probability with a fresh
frozen-model prediction on the exact recovered feature row.

In [ ]:
top_a1_features = []
if not specific_importance.empty and "feature" in specific_importance.columns:
    a1_imp = specific_importance[specific_importance["attempt"].eq(1)].copy()
    if "rank" in a1_imp.columns:
        a1_imp = a1_imp.sort_values("rank")
    top_a1_features = a1_imp["feature"].head(10).tolist()

verification, feature_values, a1_metadata = recompute_a1_candidate_probabilities(
    ART,
    decision_table,
    top_features=top_a1_features,
    event_meta=meta,
)

display(verification)
print(
    "Maximum |saved p - recomputed p| =",
    verification["abs_difference"].max(),
)

if feature_values is not None:
    print("\nValues of the top frozen A1 features for these exact route candidates:")
    display(feature_values)

### What this demonstrates

For one real transaction we can now trace:

\[
\text{transaction context}
\rightarrow
\text{candidate feature state}
\rightarrow
\text{frozen XGBoost probability}
\rightarrow
\text{route pressure}
\rightarrow
\text{adjusted policy score}
\rightarrow
\text{chosen route}
\]

The predictive model answers **which route looks best for this transaction**.

The optimizer answers **which route should be consumed now given current scarcity**.

\[
\boxed{\text{prediction}} \neq \boxed{\text{decision}}
\]

# Part XII — Robustness rather than one magic λ

## 13. Later-period robustness table

In [ ]:
rob=ART['robustness']
val_path=rob/'summary/validation_overall.csv' if rob else None
if val_path and val_path.exists():
    validation=pd.read_csv(val_path)
    display(validation)
    daily_path=rob/'summary/validation_daily_rollup.csv'
    if daily_path.exists(): display(pd.read_csv(daily_path))
else:
    print('Frozen validation headline: on Jun 8–14 the main pressure policies beat greedy on both modeled approvals and approved value on all 7/7 days.')

# Part XIII — Scientific interpretation

Following discussion with the data scientist, offline policy evaluation is centered on the model's calibrated probability estimates. That makes the comparison coherent and realistic **under the frozen model**.

But only $Y_i(R_{logged})$ is observed. $Y_i(R_{alternative})$ is counterfactual. Therefore:

- AUC/AP/log-loss/Brier/ECE → **factual predictive evidence**;
- capacity/movement → **operational evidence**;
- LP/pressure uplift → **model-implied opportunity**;
- true causal production lift → requires **prospective controlled A/B testing**.

## Claim taxonomy — what each result actually means

In [ ]:
display(pd.DataFrame([
    ["AUC / AP / log-loss / Brier / ECE", "FACTUAL", "Observed-route predictive quality"],
    ["Capacity compliance / route shares / movement", "OPERATIONAL", "Direct property of replayed assignments"],
    ["Expected approvals / approved amount on alternative routes", "MODEL-DEPENDENT", "Counterfactual estimate under frozen calibrated probabilities"],
    ["Observed success when suggestion matches logged route", "DESCRIPTIVE ONLY", "Overlap subset; not an unbiased policy-value estimate"],
], columns=["metric", "claim type", "interpretation"]))

## 14. Final validation-status table

In [ ]:
display(pd.DataFrame([
['Predictive quality','Chronologically validated on observed routes'],
['Calibration','Measured on factual route outcomes'],
['Capacity behavior','Validated in chronological replay'],
['LP / pressure gains','Counterfactual, calibrated-model-implied'],
['True production uplift','Requires controlled online/A-B test'],
],columns=['layer','status']))

## Technical submission freeze check

In [ ]:
checks = pd.DataFrame([
    ["Standardized real data", ART.get("data") is not None and Path(ART["data"]).exists()],
    ["A1 frozen metadata", ART.get("a1_metadata") is not None and Path(ART["a1_metadata"]).exists()],
    ["A2 frozen metadata", ART.get("a2_metadata") is not None and Path(ART["a2_metadata"]).exists()],
    ["A3 frozen metadata", ART.get("a3_metadata") is not None and Path(ART["a3_metadata"]).exists()],
    ["A1 factual predictions", ART.get("a1_predictions") is not None and Path(ART["a1_predictions"]).exists()],
    ["A2 factual predictions", ART.get("a2_predictions") is not None and Path(ART["a2_predictions"]).exists()],
    ["A3 factual predictions", ART.get("a3_predictions") is not None and Path(ART["a3_predictions"]).exists()],
    ["Chronological candidate backtest", ART.get("backtest") is not None and Path(ART["backtest"]).exists()],
    ["Exact A1 candidate matrix", ART.get("a1_candidate_matrix") is not None and Path(ART["a1_candidate_matrix"]).exists()],
    ["A1/A2/A3 compact configs exported", all((LOCAL_REPO/"configs"/f"a{i}_final.json").exists() for i in [1,2,3])],
], columns=["check", "passed"])

display(checks)
print("✓ Technical showcase ready to freeze" if checks["passed"].all() else "Inspect any failed artifact checks above")

# Conclusion

This is not merely an XGBoost classifier. It is a dynamic financial decision system:

**concept drift → temporal state → stage-specific probability estimation → constrained optimization → online scarcity pricing → controlled live validation.**

The main online result is that moderate capacity pressure captures roughly **85–86% of the model-implied LP opportunity**, while behaving much more sensibly with route capacity than naive greedy routing.